# Setup

In [ ]:
print("hello")

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde


import config
import hatfield_funcs as hf

from pathlib import Path
import sys

project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root / "src"))

from biofilm_data.io import (  # For loading tests
    load_groundtruth_resampled_spectra,
    load_groundtruth_ndvi_station_stats,
    load_groundtruth_ndwi_station_stats,
    load_imagery_ndvi_station_stats,
    load_imagery_station_spectra,
    load_lab_data,
)

plt.style.use("default")
%config InlineBackend.figure_format = 'retina'

In [ ]:
if os.getenv("GA_STORAGE_ACCOUNT_NAME") is None:
    raise RuntimeError(
        "This notebook is configured for Azure cloud storage. "
        "Set GA_STORAGE_ACCOUNT_NAME before running it."
    )

hf.config.configure_azure()
hf.config.configure_data_root_path(config.CONTAINER_NAME)

# Read data

In [ ]:
# Ground-truth resampled spectra and indices

df_reflectance_spectra_resampled_to_mica10b = load_groundtruth_resampled_spectra(
    config.CONTAINER_FILE_PATHS["groundtruth_resampled_spectra_csv"])

df_ndvi740_station_stats = load_groundtruth_ndvi_station_stats(config.CONTAINER_FILE_PATHS["groundtruth_ndvi_station_stats_csv"])

df_ndwi740_station_stats = load_groundtruth_ndwi_station_stats(config.CONTAINER_FILE_PATHS["groundtruth_ndwi_station_stats_csv"])

In [ ]:
# Lab data

df_lab = load_lab_data(config.CONTAINER_FILE_PATHS["lab_data_csv"])

In [ ]:
# Imagery spectra

df_imagery_spectra = load_imagery_station_spectra(config.CONTAINER_FILE_PATHS["imagery_station_spectra_csv"])

# Functions

In [ ]:
def safe_normalized_difference(
    band_a: pd.Series,
    band_b: pd.Series,
    *,
    zero_tolerance: float = 1e-8,
) -> pd.Series:
    """
    Calculate a normalized difference while avoiding unstable division.

    The normalized difference is calculated as:

    ``(band_a - band_b) / (band_a + band_b)``

    Results are ``NaN`` where either input is non-finite or where the
    denominator is within ``zero_tolerance`` of zero.

    Parameters
    ----------
    band_a, band_b : pandas.Series
        Numeric bands with matching indices.

    zero_tolerance : float, default 1e-8
        Absolute tolerance used to identify denominators near zero.

    Returns
    -------
    pandas.Series
        Normalized difference with the same index as the input Series.

    Raises
    ------
    ValueError
        If the input indices differ or ``zero_tolerance`` is invalid.
    """
    if not band_a.index.equals(band_b.index):
        raise ValueError("band_a and band_b must have matching indices.")

    if not np.isfinite(zero_tolerance) or zero_tolerance < 0:
        raise ValueError("zero_tolerance must be a finite, non-negative number.")

    band_a = band_a.astype("float64")
    band_b = band_b.astype("float64")

    numerator = band_a - band_b
    denominator = band_a + band_b

    valid = (
        np.isfinite(band_a)
        & np.isfinite(band_b)
        & ~np.isclose(
            denominator,
            0.0,
            rtol=0.0,
            atol=zero_tolerance,
        )
    )

    return numerator.div(denominator.where(valid))



# Select data subsets for R samples round 1 and complete

In [ ]:
round_of_interest = 1

df_reflectance_spectra_resampled_to_mica10b_R_r1 = (
    df_reflectance_spectra_resampled_to_mica10b
    .xs((round_of_interest, "R"), level=("round", "sample_type"))
)

df_reflectance_spectra_resampled_to_mica10b_R_r1_stats = (
    df_reflectance_spectra_resampled_to_mica10b_R_r1
    .groupby(["site", "station_id"])
    .agg(["mean", "std"])
)

df_indices_station_R_r1_stats = (
    df_ndvi740_station_stats
    .xs((round_of_interest, "R"), level=("round", "sample_type"))
    .drop(columns="scan_count")
    .join(
        (df_ndwi740_station_stats
        .xs((round_of_interest, "R"), level=("round", "sample_type"))
        .drop(columns="scan_count")),
        how="inner"
    )
)

df_lab_R_r1 = (
    df_lab
    .xs((round_of_interest, 'R'), level=('round', 'sample_type'))
)
df_lab_R_r1_fines = df_lab_R_r1["Percent Silt (%)"] + df_lab_R_r1["Percent Clay (%)"]
df_lab_R_r1_fines.name = "Fines (%)"

df_green_red_ratio_R_r1_stats = (
    (df_reflectance_spectra_resampled_to_mica10b_R_r1['560']
    / df_reflectance_spectra_resampled_to_mica10b_R_r1['668'])
    .groupby(["site", "station_id"])
    .agg(["mean", "std"])
)
df_green_red_ratio_R_r1_stats.rename(columns={"mean": "Green / Red ratio mean", "std": "Green / Red ratio std"}, inplace=True)

df_imagery_spectra_r1 = (
    df_imagery_spectra
    .xs((round_of_interest, 'DLS', 'full'), level=('round', 'dls_mode', 'resolution'))
).copy()
df_imagery_spectra_r1["ndvi"] = safe_normalized_difference(
    df_imagery_spectra_r1["740"],
    df_imagery_spectra_r1["668"],
)
df_imagery_spectra_r1["ndwi"] = safe_normalized_difference(
    df_imagery_spectra_r1["560"],
    df_imagery_spectra_r1["740"],
)
df_imagery_spectra_r1["weight"] = df_imagery_spectra_r1["polygon_coverage_fraction"]


In [ ]:
# =========================================================
# Columns for weighted mean and standard deviation
# =========================================================

weighted_stat_columns = (
    [
        str(wavelength)
        for wavelength in config.MICASENSE_10B_CENTER_WAVELENGTHS
    ]
    + [
        "ndvi",
        "ndwi",
    ]
)

# Remove possible duplicates while preserving order
weighted_stat_columns = list(
    dict.fromkeys(weighted_stat_columns)
)


# =========================================================
# Validate required columns
# =========================================================

required_columns = (
    weighted_stat_columns
    + ["weight"]
)

missing_columns = [
    column
    for column in required_columns
    if column not in df_imagery_spectra_r1.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing from "
        "df_imagery_spectra_r1:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# =========================================================
# Weighted statistics
# =========================================================

def weighted_spectral_statistics(
    group: pd.DataFrame,
    value_columns,
    weight_column: str = "weight",
) -> pd.Series:
    """
    Calculate weighted mean and standard deviation for multiple
    spectral columns using the same valid pixels.

    Returns mean/std for each variable and one common set of:
      - pixel_count
      - effective_pixel_count
      - valid_polygon_coverage_fraction
    """

    # Convert all requested variables to one numeric matrix
    values_all = (
        group[value_columns]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .to_numpy(dtype="float64")
    )

    weights_all = pd.to_numeric(
        group[weight_column],
        errors="coerce",
    ).to_numpy(dtype="float64")

    # Joint validity mask:
    # every requested value and the weight must be valid
    valid = (
        np.isfinite(values_all).all(axis=1)
        & np.isfinite(weights_all)
        & (weights_all > 0)
    )

    values = values_all[valid]
    weights = weights_all[valid]

    output = {}

    if len(values) == 0:
        for column in value_columns:
            output[f"{column}_mean"] = np.nan
            output[f"{column}_std"] = np.nan

        output["pixel_count"] = 0
        output["effective_pixel_count"] = np.nan
        output["valid_polygon_coverage_fraction"] = 0.0

        return pd.Series(output)

    # -----------------------------------------------------
    # Weighted means for all variables
    # -----------------------------------------------------

    weighted_means = np.average(
        values,
        axis=0,
        weights=weights,
    )

    # -----------------------------------------------------
    # Area-weighted population variances and standard
    # deviations for all variables
    # -----------------------------------------------------

    weighted_variances = np.average(
        (values - weighted_means) ** 2,
        axis=0,
        weights=weights,
    )

    weighted_stds = np.sqrt(
        weighted_variances
    )

    # -----------------------------------------------------
    # Store mean and standard deviation for each variable
    # -----------------------------------------------------

    for column, mean, std in zip(
        value_columns,
        weighted_means,
        weighted_stds,
    ):
        output[f"{column}_mean"] = mean
        output[f"{column}_std"] = std

    # -----------------------------------------------------
    # Common pixel and coverage statistics
    # -----------------------------------------------------

    output["pixel_count"] = len(values)

    output["effective_pixel_count"] = (
        weights.sum() ** 2
        / np.square(weights).sum()
    )

    output[
        "valid_polygon_coverage_fraction"
    ] = weights.sum()

    return pd.Series(output)


# =========================================================
# Polygon index levels
# =========================================================

polygon_index_levels = [
    level
    for level in df_imagery_spectra_r1.index.names
    if level != "pixel_id"
]

if not polygon_index_levels:
    raise ValueError(
        "No polygon index levels remain after excluding pixel_id."
    )


# =========================================================
# Calculate polygon-level statistics
# =========================================================

df_imagery_spectra_r1_stats = (
    df_imagery_spectra_r1
    .groupby(
        level=polygon_index_levels,
        sort=False,
        dropna=False,
    )
    .apply(
        weighted_spectral_statistics,
        value_columns=weighted_stat_columns,
        weight_column="weight",
    )
)

df_imagery_spectra_r1_stats.head()

<!-- # Setup -->

# Redo Trinity plots about NDWI but with ground-truth data

## Mean selected band values at stations

In [ ]:
# =========================================================
# Band definitions
# =========================================================

bands = [
    ('475', "Blue (475 nm)"),
    ('560', "Green (560 nm)"),
    ('668', "Red (668 nm)"),
    ('740', "NIR (740 nm)"),
]


# =========================================================
# Identify available sites
# =========================================================

available_sites = (
    df_reflectance_spectra_resampled_to_mica10b_R_r1_stats.index.get_level_values(
        "site"
    ).unique()
)

# Preserve the order defined in config.SITES
sites = [site for site in config.SITES if site in available_sites]

if not sites:
    raise ValueError("None of the sites in config.SITES were found in the DataFrame.")


# =========================================================
# Validate wavelengths
# =========================================================

available_wavelengths = set(
    df_reflectance_spectra_resampled_to_mica10b_R_r1_stats.columns.get_level_values(0)
)

missing_wavelengths = [
    wavelength for wavelength, _ in bands if wavelength not in available_wavelengths
]

if missing_wavelengths:
    raise KeyError(
        "The following wavelengths are missing from the DataFrame: "
        f"{missing_wavelengths}"
    )


# =========================================================
# Plot settings
# =========================================================

fig, ax = plt.subplots(figsize=(10, 6))

band_positions = np.arange(len(bands))

# Total horizontal space occupied by site boxes within each band
group_width = 0.75

# Width allocated to each site
site_spacing = group_width / len(sites)
box_width = site_spacing * 0.75

# Center the site boxes around each band position
site_offsets = (np.arange(len(sites)) - (len(sites) - 1) / 2) * site_spacing

rng = np.random.default_rng(42)


# =========================================================
# Create boxplots and individual points
# =========================================================

for site_number, site in enumerate(sites):
    site_color = config.SITE_COLORS.get(
        site,
        "gray",
    )

    site_label = config.SITE_LABELS.get(
        site,
        site,
    )

    site_data = df_reflectance_spectra_resampled_to_mica10b_R_r1_stats.xs(
        site,
        level="site",
    )

    # Add the site to the legend only once
    site_label_added = False

    for band_number, (wavelength, band_label) in enumerate(bands):
        values = site_data[(wavelength, "mean")].dropna().to_numpy()

        if len(values) == 0:
            continue

        position = band_positions[band_number] + site_offsets[site_number]

        # Draw one box for this site and band
        ax.boxplot(
            [values],
            positions=[position],
            widths=box_width,
            patch_artist=True,
            showmeans=True,
            manage_ticks=False,
            boxprops={
                "facecolor": site_color,
                "edgecolor": site_color,
                "alpha": 0.35,
                "linewidth": 1.5,
            },
            medianprops={
                "color": site_color,
                "linewidth": 2,
            },
            meanprops={
                "marker": "D",
                "markerfacecolor": "white",
                "markeredgecolor": site_color,
                "markersize": 5,
            },
            whiskerprops={
                "color": site_color,
                "linewidth": 1.2,
            },
            capprops={
                "color": site_color,
                "linewidth": 1.2,
            },
            flierprops={
                "marker": "none",
            },
        )

        # Slight horizontal jitter around the corresponding box
        jitter = rng.uniform(
            -box_width * 0.3,
            box_width * 0.3,
            size=len(values),
        )

        ax.scatter(
            position + jitter,
            values,
            color=site_color,
            edgecolor="black",
            linewidth=0.4,
            s=28,
            alpha=0.75,
            zorder=3,
            label=(site_label if not site_label_added else None),
        )

        site_label_added = True


# =========================================================
# Format axes
# =========================================================

ax.set_xticks(
    band_positions,
    labels=[label for _, label in bands],
)

ax.set_xlabel("Spectral band")
ax.set_ylabel("Station mean reflectance")

# Retain horizontal reference lines only
ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.3,
)

# Explicitly disable vertical lines between bands
ax.grid(
    False,
    axis="x",
)

# Remove x-axis tick marks while retaining the labels
ax.tick_params(
    axis="x",
    length=0,
)

ax.legend()

fig.tight_layout()
plt.show()

## Green vs red band mean values

In [ ]:
# Green vs red bands

fig, ax = plt.subplots()
for site in config.SITES:
    df_reflectance_spectra_resampled_to_mica10b_R_r1_stats_site = (
        df_reflectance_spectra_resampled_to_mica10b_R_r1_stats.xs(site, level="site")
    )
    ax.errorbar(
        df_reflectance_spectra_resampled_to_mica10b_R_r1_stats_site['668']["mean"],
        df_reflectance_spectra_resampled_to_mica10b_R_r1_stats_site['560']["mean"],
        xerr=df_reflectance_spectra_resampled_to_mica10b_R_r1_stats_site['668']['std'],
        yerr=df_reflectance_spectra_resampled_to_mica10b_R_r1_stats_site['560']['std'],
        color=config.SITE_COLORS.get(site, "k"), fmt='none', alpha=0.3,
    )
    ax.scatter(
        df_reflectance_spectra_resampled_to_mica10b_R_r1_stats_site['668']["mean"],
        df_reflectance_spectra_resampled_to_mica10b_R_r1_stats_site['560']["mean"],
        color=config.SITE_COLORS.get(site, "k"),
        label=config.SITE_LABELS.get(site, site),
    )
lims = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1]),
]
ax.plot(
    lims,
    lims,
    color="black",
    linestyle="--",
    linewidth=1,
    label="1:1 line",
)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Red band (668 nm) mean reflectance")
ax.set_ylabel("Green band (560 nm) mean reflectance")
ax.legend()

## NDWI vs. NDVI

In [ ]:
# NDWI vs. NDVI

fig, ax = plt.subplots()
for site in config.SITES:
    df_indices_station_R_r1_stats_site = df_indices_station_R_r1_stats.xs(
        site, level="site"
    )
    ax.errorbar(
        df_indices_station_R_r1_stats_site["ndvi_mean"],
        df_indices_station_R_r1_stats_site["ndwi_mean"],
        xerr=df_indices_station_R_r1_stats_site["ndvi_std"],
        yerr=df_indices_station_R_r1_stats_site["ndwi_std"],
        color=config.SITE_COLORS.get(site, "k"),
        fmt="none",
        alpha=0.3,
    )
    ax.scatter(
        df_indices_station_R_r1_stats_site["ndvi_mean"],
        df_indices_station_R_r1_stats_site["ndwi_mean"],
        color=config.SITE_COLORS.get(site, "k"),
        label=config.SITE_LABELS.get(site, site),
    )
lims = [
    min(ax.get_xlim()[0], -ax.get_ylim()[1]),
    max(ax.get_xlim()[1], -ax.get_ylim()[0]),
]
ax.plot(
    lims,
    [-lims[0], -lims[1]],
    color="black",
    linestyle="--",
    linewidth=1,
    label="Slope −1 line",
)
ax.set_xlim(lims)
ax.set_ylim([-lims[1], -lims[0]])
ax.set_xlabel("Mean NDVI")
ax.set_ylabel("Mean NDWI")
ax.legend()

## Green / Red ratio vs Fines %

In [ ]:
# Green / Red ratio vs Fines %

df_plot = df_green_red_ratio_R_r1_stats.join(df_lab_R_r1_fines, how="inner")

fig, ax = plt.subplots()
for site in config.SITES:
    df_plot_site = df_plot.xs(
        site, level="site"
    )
    ax.errorbar(
        df_plot_site['Fines (%)'],
        df_plot_site["Green / Red ratio mean"],
        xerr=None,
        yerr=df_plot_site["Green / Red ratio std"],
        color=config.SITE_COLORS.get(site, "k"),
        fmt="none",
        alpha=0.3,
    )
    ax.scatter(
        df_plot_site['Fines (%)'],
        df_plot_site["Green / Red ratio mean"],
        color=config.SITE_COLORS.get(site, "k"),
        label=config.SITE_LABELS.get(site, site),
    )
# lims = [
#     min(ax.get_xlim()[0], ax.get_ylim()[0]),
#     max(ax.get_xlim()[1], ax.get_ylim()[1]),
# ]
# ax.plot(
#     lims,
#     lims,
#     color="black",
#     linestyle="--",
#     linewidth=1,
#     label="1:1 line",
# )
# ax.set_xlim(lims)
# ax.set_ylim(lims)
# ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Top 2 mm fines (%)")
ax.set_ylabel("Green (560 nm) / Red (668 nm) ratio")
ax.legend()

# Ground-truth vs. imagery or imagery alone

In [ ]:
# =========================================================
# Combine imagery and ground-truth indices
# =========================================================

df_indices = (
    (
        df_imagery_spectra_r1_stats[['ndvi_mean', 'ndvi_std', 'ndwi_mean', 'ndwi_std']]
        .rename(columns=lambda column: f"imagery_{column}")
        .reset_index()
        .set_index(
            [
                "site",
                "station_id",
            ]
        )
    )
    .join((
        df_indices_station_R_r1_stats[['ndvi_mean', 'ndvi_std', 'ndwi_mean', 'ndwi_std']]
        .rename(columns=lambda column: f"gt_{column}")
    )
    , how="inner")
    .reset_index()
    .set_index(
        [
            "site",
            "station_id",
            "sampled_after_flight",
        ]
    )
)

# =========================================================
# Marker size definitions
# =========================================================

# sampled_after_flight=False:
# sampled before imagery acquisition, so potentially trampled
small_marker_size = 15

# sampled_after_flight=True:
# sampled after imagery acquisition, so not trampled during flight
large_marker_size = 90

marker_sizes = {
    False: small_marker_size,
    True: large_marker_size,
}

sampling_labels = {
    False: "Trampled — sampled before flight",
    True: "Not trampled — sampled after flight",
}


# =========================================================
# Available sites
# =========================================================

available_sites = set(df_indices.index.get_level_values("site"))


# =========================================================
# Create NDVI and NDWI plots
# =========================================================

for ind in [
    "ndvi",
    "ndwi",
]:
    fig, ax = plt.subplots(figsize=(7, 7))

    for site in config.SITES:
        if site not in available_sites:
            continue

        site_color = config.SITE_COLORS.get(
            site,
            "k",
        )

        df_indices_site = df_indices.xs(
            site,
            level="site",
        )

        # -------------------------------------------------
        # Error bars for every station
        # -------------------------------------------------

        ax.errorbar(
            df_indices_site[f"gt_{ind}_mean"],
            df_indices_site[f"imagery_{ind}_mean"],
            xerr=df_indices_site[f"gt_{ind}_std"],
            yerr=df_indices_site[f"imagery_{ind}_std"],
            color=site_color,
            fmt="none",
            alpha=0.3,
            linewidth=1,
            zorder=1,
        )

        # -------------------------------------------------
        # Separate trampled and non-trampled points
        # -------------------------------------------------

        sampled_after_values = df_indices_site.index.get_level_values(
            "sampled_after_flight"
        )

        for sampled_after_flight in [
            False,
            True,
        ]:
            status_mask = sampled_after_values == sampled_after_flight

            df_status = df_indices_site.loc[status_mask]

            if df_status.empty:
                continue

            ax.scatter(
                df_status[f"gt_{ind}_mean"],
                df_status[f"imagery_{ind}_mean"],
                color=site_color,
                edgecolor="black",
                facecolor=site_color,
                linewidth=0.5,
                s=marker_sizes[sampled_after_flight],
                marker="o",
                alpha=0.85,
                zorder=2,
            )

    # =====================================================
    # Add 1:1 line
    # =====================================================

    lims = [
        min(
            ax.get_xlim()[0],
            ax.get_ylim()[0],
        ),
        max(
            ax.get_xlim()[1],
            ax.get_ylim()[1],
        ),
    ]

    (one_to_one_line,) = ax.plot(
        lims,
        lims,
        color="black",
        linestyle="--",
        linewidth=1,
        label="1:1 line",
        zorder=0,
    )

    ax.set_xlim(lims)
    ax.set_ylim(lims)

    ax.set_aspect(
        "equal",
        adjustable="box",
    )

    # =====================================================
    # Axis labels
    # =====================================================

    ax.set_xlabel(f"Ground truth {ind.upper()} mean")

    ax.set_ylabel(f"Imagery {ind.upper()} mean")

    ax.set_title(f"{ind.upper()} comparison")

    # =====================================================
    # Site-color legend
    # =====================================================

    site_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="none",
            markerfacecolor=config.SITE_COLORS.get(
                site,
                "k",
            ),
            markeredgecolor="black",
            markeredgewidth=0.5,
            markersize=7,
            label=config.SITE_LABELS.get(
                site,
                site,
            ),
        )
        for site in config.SITES
        if site in available_sites
    ]

    site_legend = ax.legend(
        handles=site_handles,
        # title="Site",
        loc="upper left",
    )

    # Keep the site legend when adding the second legend
    ax.add_artist(site_legend)

    # =====================================================
    # Marker-size and reference-line legend
    # =====================================================

    sampling_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="none",
            markerfacecolor="gray",
            markeredgecolor="black",
            markeredgewidth=0.5,
            markersize=np.sqrt(marker_sizes[sampled_after_flight]),
            label=sampling_labels[sampled_after_flight],
        )
        for sampled_after_flight in [
            False,
            True,
        ]
    ]

    sampling_handles.append(one_to_one_line)

    ax.legend(
        handles=sampling_handles,
        # title="Sampling status",
        loc="lower right",
    )

    ax.grid(
        linestyle="--",
        alpha=0.2,
    )

    fig.tight_layout()
    plt.show()


In [ ]:
# =========================================================
# Settings
# =========================================================

x_index = "ndvi"
y_index = "ndwi"

small_marker_size = 30
large_marker_size = 90

marker_sizes = {
    False: small_marker_size,
    True: large_marker_size,
}

sampling_labels = {
    False: "Trampled — sampled before flight",
    True: "Not trampled — sampled after flight",
}

available_sites = set(
    df_indices.index.get_level_values("site")
)


# =========================================================
# Create plot
# =========================================================

fig, ax = plt.subplots(
    figsize=(6, 5.5)
)

for site in config.SITES:

    if site not in available_sites:
        continue

    site_color = config.SITE_COLORS.get(
        site,
        "k",
    )

    df_site = df_indices.xs(
        site,
        level="site",
    )

    # -----------------------------------------------------
    # NDVI and NDWI error bars
    # -----------------------------------------------------

    ax.errorbar(
        df_site[f"imagery_{x_index}_mean"],
        df_site[f"imagery_{y_index}_mean"],
        xerr=df_site[f"imagery_{x_index}_std"],
        yerr=df_site[f"imagery_{y_index}_std"],
        color=site_color,
        fmt="none",
        linewidth=1,
        alpha=0.3,
        zorder=1,
    )

    # -----------------------------------------------------
    # Separate by trampling status
    # -----------------------------------------------------

    sampled_after_values = (
        df_site.index.get_level_values(
            "sampled_after_flight"
        )
    )

    for sampled_after_flight in [
        False,
        True,
    ]:
        status_mask = (
            sampled_after_values
            == sampled_after_flight
        )

        df_status = df_site.loc[
            status_mask
        ]

        if df_status.empty:
            continue

        ax.scatter(
            df_status[f"imagery_{x_index}_mean"],
            df_status[f"imagery_{y_index}_mean"],
            color=site_color,
            edgecolor="black",
            linewidth=0.5,
            marker="o",
            s=marker_sizes[
                sampled_after_flight
            ],
            alpha=0.85,
            zorder=2,
        )


# =========================================================
# Add y = -x reference line
# =========================================================

lims = np.array(
    [
        min(
            ax.get_xlim()[0],
            -ax.get_ylim()[1],
        ),
        max(
            ax.get_xlim()[1],
            -ax.get_ylim()[0],
        ),
    ]
)

slope_minus_one_line, = ax.plot(
    lims,
    -lims,
    color="black",
    linestyle="--",
    linewidth=1,
    label="Slope -1 line",
    zorder=0,
)

# Matching limits for a slope -1 line
ax.set_xlim(lims)
ax.set_ylim(-lims[::-1])
ax.set_aspect(
    "equal",
    adjustable="box",
)


# =========================================================
# Axis labels
# =========================================================

ax.set_xlabel(
    "Imagery NDVI mean"
)

ax.set_ylabel(
    "Imagery NDWI mean"
)

ax.grid(
    linestyle="--",
    alpha=0.2,
)


# =========================================================
# Site legend
# =========================================================

site_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markerfacecolor=config.SITE_COLORS.get(
            site,
            "k",
        ),
        markeredgecolor="black",
        markeredgewidth=0.5,
        markersize=7,
        label=config.SITE_LABELS.get(
            site,
            site,
        ),
    )
    for site in config.SITES
    if site in available_sites
]

site_legend = ax.legend(
    handles=site_handles,
    # title="Site",
    loc="upper right",
)

ax.add_artist(site_legend)


# =========================================================
# Trampling-status legend
# =========================================================

status_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markerfacecolor="gray",
        markeredgecolor="black",
        markeredgewidth=0.5,
        markersize=np.sqrt(
            marker_sizes[
                sampled_after_flight
            ]
        ),
        label=sampling_labels[
            sampled_after_flight
        ],
    )
    for sampled_after_flight in [
        False,
        True,
    ]
]

status_handles.append(
    slope_minus_one_line
)

ax.legend(
    handles=status_handles,
    # title="Sampling status",
    loc="lower left",
)

fig.tight_layout()
plt.show()

In [ ]:
df_band_stats = (
    df_reflectance_spectra_resampled_to_mica10b_R_r1_stats
    .copy()
)

df_band_stats.columns = [
    f"gt_{wavelength}_{statistic}"
    for wavelength, statistic
    in df_band_stats.columns.to_flat_index()
]

df_band_stats = (
    df_band_stats.join(
        (
            df_imagery_spectra_r1_stats
            [[
                f"{wv}_{stat}"
                for wv in config.MICASENSE_10B_CENTER_WAVELENGTHS
                for stat in ["mean", "std"]
            ]]
            .rename(columns=lambda column: f"imagery_{column}")
            .reset_index()
            .set_index(['site', 'station_id'])
        ),
        how='inner'
    )
    .reset_index()
    .set_index(['site', 'station_id', 'sampled_after_flight'])
)

df_band_stats.head()

In [ ]:

# =========================================================
# Plot options
# =========================================================

# False: show only the grey all-band fit.
# True: show the grey all-band fit and the individual
#       blue, green, red, and NIR fits.
SHOW_INDIVIDUAL_BAND_FITS = False


# =========================================================
# Band definitions
# =========================================================

focus_bands = {
    "475": {
        "label": "Blue (475 nm)",
        "color": "#0072B2",
    },
    "560": {
        "label": "Green (560 nm)",
        "color": "#009E73",
    },
    "668": {
        "label": "Red (668 nm)",
        "color": "#D62728",
    },
    "740": {
        "label": "NIR (740 nm)",
        "color": "#8C564B",
    },
}

all_bands = [
    str(wavelength)
    for wavelength in config.MICASENSE_10B_CENTER_WAVELENGTHS
]


# =========================================================
# Markers and colors
# =========================================================

available_markers = [
    "o",
    "s",
    "^",
    "v",
    "D",
    "P",
    "X",
    "<",
    ">",
    "*",
    "h",
    "d",
]

band_markers = {
    wavelength: available_markers[
        band_number % len(available_markers)
    ]
    for band_number, wavelength in enumerate(all_bands)
}

# Focus bands use individual colors.
# Remaining bands are grey.
band_colors = {
    wavelength: (
        focus_bands[wavelength]["color"]
        if wavelength in focus_bands
        else "0.55"
    )
    for wavelength in all_bands
}

band_labels = {
    wavelength: (
        focus_bands[wavelength]["label"]
        if wavelength in focus_bands
        else f"{wavelength} nm"
    )
    for wavelength in all_bands
}


# =========================================================
# Validate required columns
# =========================================================

required_columns = []

for wavelength in all_bands:
    required_columns.extend(
        [
            f"gt_{wavelength}_mean",
            f"gt_{wavelength}_std",
            f"imagery_{wavelength}_mean",
            f"imagery_{wavelength}_std",
        ]
    )

missing_columns = [
    column
    for column in required_columns
    if column not in df_band_stats.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing from "
        "df_band_stats:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# =========================================================
# Identify available sites
# =========================================================

available_sites = set(
    df_band_stats.index.get_level_values("site")
)

sites = [
    site
    for site in config.SITES
    if site in available_sites
]

if not sites:
    raise ValueError(
        "None of the sites in config.SITES were found in df_band_stats."
    )


# =========================================================
# Linear-fit helper
# =========================================================

def calculate_linear_fit(x, y):
    """
    Calculate a first-order least-squares fit and R².
    """

    x = np.asarray(
        x,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if (
        len(x) < 2
        or np.unique(x).size < 2
    ):
        return None

    slope, intercept = np.polyfit(
        x,
        y,
        deg=1,
    )

    predicted = (
        slope * x
        + intercept
    )

    residual_sum_squares = np.sum(
        (y - predicted) ** 2
    )

    total_sum_squares = np.sum(
        (y - y.mean()) ** 2
    )

    if total_sum_squares > 0:
        r_squared = (
            1
            - residual_sum_squares
            / total_sum_squares
        )
    else:
        r_squared = np.nan

    return {
        "slope": slope,
        "intercept": intercept,
        "r_squared": r_squared,
        "n": len(x),
        "x": x,
        "y": y,
    }


# =========================================================
# Create one subplot per site
# =========================================================

fig, axes = plt.subplots(
    nrows=1,
    ncols=len(sites),
    figsize=(6 * len(sites), 5.5),
    sharex=True,
    sharey=True,
    squeeze=False,
)

axes = axes.ravel()


# Plot grey bands first so focus bands remain visible
plot_order = (
    [
        wavelength
        for wavelength in all_bands
        if wavelength not in focus_bands
    ]
    + [
        wavelength
        for wavelength in all_bands
        if wavelength in focus_bands
    ]
)


# Store fitted coefficients and R² values
fit_results = []


# =========================================================
# Plot each site
# =========================================================

for ax, site in zip(axes, sites):

    df_site = df_band_stats.xs(
        site,
        level="site",
    )

    # Accumulate every wavelength/station point for the
    # combined all-band fit.
    all_fit_x = []
    all_fit_y = []

    for wavelength in plot_order:

        x_column = f"gt_{wavelength}_mean"
        y_column = f"imagery_{wavelength}_mean"
        xerr_column = f"gt_{wavelength}_std"
        yerr_column = f"imagery_{wavelength}_std"

        x_values = pd.to_numeric(
            df_site[x_column],
            errors="coerce",
        )

        y_values = pd.to_numeric(
            df_site[y_column],
            errors="coerce",
        )

        x_errors = pd.to_numeric(
            df_site[xerr_column],
            errors="coerce",
        )

        y_errors = pd.to_numeric(
            df_site[yerr_column],
            errors="coerce",
        )

        valid = (
            x_values.notna()
            & y_values.notna()
        )

        if not valid.any():
            continue

        x_values = x_values.loc[valid]
        y_values = y_values.loc[valid]

        # Missing standard deviations produce zero-length
        # error bars rather than dropping a point.
        x_errors = (
            x_errors.loc[valid]
            .fillna(0)
        )

        y_errors = (
            y_errors.loc[valid]
            .fillna(0)
        )

        x_array = x_values.to_numpy(
            dtype=float
        )

        y_array = y_values.to_numpy(
            dtype=float
        )

        # Add every wavelength to the pooled all-band fit.
        all_fit_x.append(x_array)
        all_fit_y.append(y_array)

        band_color = band_colors[
            wavelength
        ]

        band_marker = band_markers[
            wavelength
        ]

        is_focus_band = (
            wavelength in focus_bands
        )

        # -------------------------------------------------
        # Error bars
        # -------------------------------------------------

        ax.errorbar(
            x_values,
            y_values,
            xerr=x_errors,
            yerr=y_errors,
            fmt="none",
            ecolor=band_color,
            elinewidth=1,
            alpha=(
                0.35
                if is_focus_band
                else 0.15
            ),
            zorder=(
                3
                if is_focus_band
                else 1
            ),
        )

        # -------------------------------------------------
        # Mean points
        # -------------------------------------------------

        ax.scatter(
            x_values,
            y_values,
            marker=band_marker,
            color=band_color,
            edgecolor="black",
            linewidth=0.5,
            s=(
                65
                if is_focus_band
                else 40
            ),
            alpha=(
                0.65
                if is_focus_band
                else 0.30
            ),
            zorder=(
                4
                if is_focus_band
                else 2
            ),
        )

        # -------------------------------------------------
        # Calculate each focus-band fit
        # -------------------------------------------------

        if is_focus_band:
            fit = calculate_linear_fit(
                x_array,
                y_array,
            )

            if fit is not None:

                # Extend to x=0 to display the intercept.
                x_fit = np.linspace(
                    0,
                    max(
                        0,
                        fit["x"].max(),
                    ),
                    100,
                )

                y_fit = (
                    fit["slope"] * x_fit
                    + fit["intercept"]
                )

                # Plot individual focus-band fits only
                # when the option is enabled.
                if SHOW_INDIVIDUAL_BAND_FITS:
                    ax.plot(
                        x_fit,
                        y_fit,
                        color=band_color,
                        linestyle="-",
                        linewidth=2,
                        alpha=0.9,
                        zorder=4,
                    )

                # Store results even when the line is hidden.
                fit_results.append(
                    {
                        "site": site,
                        "wavelength": wavelength,
                        "band": band_labels[
                            wavelength
                        ],
                        "fit_type": "individual band",
                        "line_displayed": (
                            SHOW_INDIVIDUAL_BAND_FITS
                        ),
                        "n": fit["n"],
                        "slope": fit["slope"],
                        "intercept": fit["intercept"],
                        "r_squared": fit["r_squared"],
                    }
                )

    # -----------------------------------------------------
    # Combined grey fit using all wavelengths
    # -----------------------------------------------------

    if all_fit_x:
        pooled_x = np.concatenate(
            all_fit_x
        )

        pooled_y = np.concatenate(
            all_fit_y
        )

        pooled_fit = calculate_linear_fit(
            pooled_x,
            pooled_y,
        )

        if pooled_fit is not None:

            pooled_x_fit = np.linspace(
                0,
                max(
                    0,
                    pooled_fit["x"].max(),
                ),
                200,
            )

            pooled_y_fit = (
                pooled_fit["slope"]
                * pooled_x_fit
                + pooled_fit["intercept"]
            )

            # The pooled grey fit is always shown.
            ax.plot(
                pooled_x_fit,
                pooled_y_fit,
                color="0.25",
                linestyle="--",
                linewidth=2.5,
                alpha=0.9,
                zorder=5,
            )

            fit_results.append(
                {
                    "site": site,
                    "wavelength": "all",
                    "band": "All bands combined",
                    "fit_type": "all bands",
                    "line_displayed": True,
                    "n": pooled_fit["n"],
                    "slope": pooled_fit["slope"],
                    "intercept": pooled_fit["intercept"],
                    "r_squared": pooled_fit["r_squared"],
                }
            )

    # Site label
    ax.set_title(
        config.SITE_LABELS.get(
            site,
            site,
        )
    )

    ax.grid(
        linestyle="--",
        alpha=0.2,
    )


# =========================================================
# Common limits
#
# The x-axis starts at exactly zero.
# The y-axis includes zero and any negative fitted intercept.
# =========================================================

common_x_max = max(
    0,
    *[
        ax.get_xlim()[1]
        for ax in axes
    ],
)

common_y_min = min(
    0,
    *[
        ax.get_ylim()[0]
        for ax in axes
    ],
)

common_y_max = max(
    0,
    *[
        ax.get_ylim()[1]
        for ax in axes
    ],
)


# Only add padding to the upper x-limit.
x_range = common_x_max

x_padding = (
    x_range * 0.02
    if x_range > 0
    else 0.01
)


# Add a small amount of vertical padding.
y_range = (
    common_y_max
    - common_y_min
)

y_padding = (
    y_range * 0.03
    if y_range > 0
    else 0.01
)


x_lower = 0

x_upper = (
    common_x_max
    + x_padding
)

y_lower = (
    common_y_min - y_padding
    if common_y_min < 0
    else 0
)

y_upper = (
    common_y_max
    + y_padding
)


for ax in axes:
    ax.set_xlim(
        x_lower,
        x_upper,
    )

    ax.set_ylim(
        y_lower,
        y_upper,
    )


# =========================================================
# Shared axis labels
# =========================================================

fig.supxlabel(
    "Ground-truth mean reflectance",
    y=0.035,
)

fig.supylabel(
    "Imagery mean reflectance",
    x=0.02,
)


# =========================================================
# Legend above the plots
# =========================================================

band_legend_handles = []

for wavelength in all_bands:

    is_focus_band = (
        wavelength in focus_bands
    )

    show_line_in_legend = (
        is_focus_band
        and SHOW_INDIVIDUAL_BAND_FITS
    )

    band_legend_handles.append(
        Line2D(
            [0],
            [0],
            marker=band_markers[
                wavelength
            ],
            linestyle=(
                "-"
                if show_line_in_legend
                else "none"
            ),
            color=band_colors[
                wavelength
            ],
            markerfacecolor=band_colors[
                wavelength
            ],
            markeredgecolor="black",
            markeredgewidth=0.5,
            linewidth=(
                2
                if show_line_in_legend
                else 0
            ),
            markersize=(
                8
                if is_focus_band
                else 6
            ),
            alpha=(
                0.9
                if is_focus_band
                else 0.5
            ),
            label=band_labels[
                wavelength
            ],
        )
    )


# Grey pooled fit handle
band_legend_handles.append(
    Line2D(
        [0],
        [0],
        color="0.25",
        linestyle="--",
        linewidth=2.5,
        label="All bands combined fit",
    )
)


if SHOW_INDIVIDUAL_BAND_FITS:
    legend_title = (
        "Spectral band — solid lines are band fits; "
        "dashed line is the all-band fit"
    )
else:
    legend_title = (
        "Spectral band — dashed line is the all-band fit"
    )


fig.legend(
    handles=band_legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.99),
    ncol=6,
    title=legend_title,
)


# Manually control margins to avoid excessive spacing
# beneath the shared x-axis label.
fig.subplots_adjust(
    left=0.08,
    right=0.98,
    bottom=0.11,
    top=0.76,
    wspace=0.20,
)

plt.show()


# =========================================================
# Display fitted coefficients
# =========================================================

df_band_fit_results = (
    pd.DataFrame(fit_results)
    .sort_values(
        [
            "site",
            "fit_type",
            "wavelength",
        ]
    )
    .reset_index(drop=True)
)

display(df_band_fit_results)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# =========================================================
# Plot options
# =========================================================

# False: show only the grey all-band fit.
# True: also show the individual blue, green, red, and NIR fits.
SHOW_INDIVIDUAL_BAND_FITS = False


# Marker size indicates sampling/trampling status:
#
# False = sampled before flight, potentially trampled
# True  = sampled after flight, not trampled during imagery
MARKER_SIZES = {
    False: 35,
    True: 100,
}

SAMPLING_LABELS = {
    False: "Trampled — sampled before flight",
    True: "Not trampled — sampled after flight",
}


# =========================================================
# Band definitions
# =========================================================

focus_bands = {
    "475": {
        "label": "Blue (475 nm)",
        "color": "#0072B2",
    },
    "560": {
        "label": "Green (560 nm)",
        "color": "#009E73",
    },
    "668": {
        "label": "Red (668 nm)",
        "color": "#D62728",
    },
    "740": {
        "label": "NIR (740 nm)",
        "color": "#8C564B",
    },
}

all_bands = [
    str(wavelength)
    for wavelength in config.MICASENSE_10B_CENTER_WAVELENGTHS
]


# =========================================================
# Assign one marker per spectral band
# =========================================================

available_markers = [
    "o",
    "s",
    "^",
    "v",
    "D",
    "P",
    "X",
    "<",
    ">",
    "*",
    "h",
    "d",
]

band_markers = {
    wavelength: available_markers[
        band_number % len(available_markers)
    ]
    for band_number, wavelength in enumerate(all_bands)
}


# Focus bands use individual colors.
# Remaining bands are grey.
band_colors = {
    wavelength: (
        focus_bands[wavelength]["color"]
        if wavelength in focus_bands
        else "0.55"
    )
    for wavelength in all_bands
}

band_labels = {
    wavelength: (
        focus_bands[wavelength]["label"]
        if wavelength in focus_bands
        else f"{wavelength} nm"
    )
    for wavelength in all_bands
}


# =========================================================
# Validate required index levels
# =========================================================

required_index_levels = [
    "site",
    "sampled_after_flight",
]

missing_index_levels = [
    level
    for level in required_index_levels
    if level not in df_band_stats.index.names
]

if missing_index_levels:
    raise ValueError(
        "The following required index levels are missing "
        "from df_band_stats:\n"
        f"{missing_index_levels}"
    )


# =========================================================
# Validate required columns
# =========================================================

required_columns = []

for wavelength in all_bands:
    required_columns.extend(
        [
            f"gt_{wavelength}_mean",
            f"gt_{wavelength}_std",
            f"imagery_{wavelength}_mean",
            f"imagery_{wavelength}_std",
        ]
    )

missing_columns = [
    column
    for column in required_columns
    if column not in df_band_stats.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing from "
        "df_band_stats:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# =========================================================
# Identify available sites
# =========================================================

available_sites = set(
    df_band_stats.index.get_level_values(
        "site"
    )
)

sites = [
    site
    for site in config.SITES
    if site in available_sites
]

if not sites:
    raise ValueError(
        "None of the sites in config.SITES were found "
        "in df_band_stats."
    )


# =========================================================
# Linear-fit helper
# =========================================================

def calculate_linear_fit(x, y):
    """
    Calculate a first-order least-squares fit and R².
    """

    x = np.asarray(
        x,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if (
        len(x) < 2
        or np.unique(x).size < 2
    ):
        return None

    slope, intercept = np.polyfit(
        x,
        y,
        deg=1,
    )

    predicted = (
        slope * x
        + intercept
    )

    residual_sum_squares = np.sum(
        (y - predicted) ** 2
    )

    total_sum_squares = np.sum(
        (y - y.mean()) ** 2
    )

    if total_sum_squares > 0:
        r_squared = (
            1
            - residual_sum_squares
            / total_sum_squares
        )
    else:
        r_squared = np.nan

    return {
        "slope": slope,
        "intercept": intercept,
        "r_squared": r_squared,
        "n": len(x),
        "x": x,
        "y": y,
    }


# =========================================================
# Create one subplot per site
# =========================================================

fig, axes = plt.subplots(
    nrows=1,
    ncols=len(sites),
    figsize=(6 * len(sites), 5.5),
    sharex=True,
    sharey=True,
    squeeze=False,
)

axes = axes.ravel()


# Plot grey bands first so focus bands remain visible
plot_order = (
    [
        wavelength
        for wavelength in all_bands
        if wavelength not in focus_bands
    ]
    + [
        wavelength
        for wavelength in all_bands
        if wavelength in focus_bands
    ]
)


# Store fitted coefficients and R² values
fit_results = []


# =========================================================
# Plot each site
# =========================================================

for ax, site in zip(axes, sites):

    df_site = df_band_stats.xs(
        site,
        level="site",
    )

    # Accumulate every wavelength/station point for the
    # combined all-band fit.
    all_fit_x = []
    all_fit_y = []

    for wavelength in plot_order:

        x_column = f"gt_{wavelength}_mean"
        y_column = f"imagery_{wavelength}_mean"
        xerr_column = f"gt_{wavelength}_std"
        yerr_column = f"imagery_{wavelength}_std"

        x_values = pd.to_numeric(
            df_site[x_column],
            errors="coerce",
        )

        y_values = pd.to_numeric(
            df_site[y_column],
            errors="coerce",
        )

        x_errors = pd.to_numeric(
            df_site[xerr_column],
            errors="coerce",
        )

        y_errors = pd.to_numeric(
            df_site[yerr_column],
            errors="coerce",
        )

        valid = (
            x_values.notna()
            & y_values.notna()
        )

        if not valid.any():
            continue

        x_values = x_values.loc[valid]
        y_values = y_values.loc[valid]

        # Missing standard deviations produce zero-length
        # error bars rather than dropping a point.
        x_errors = (
            x_errors.loc[valid]
            .fillna(0)
        )

        y_errors = (
            y_errors.loc[valid]
            .fillna(0)
        )

        x_array = x_values.to_numpy(
            dtype=float
        )

        y_array = y_values.to_numpy(
            dtype=float
        )

        # Add every wavelength to the pooled all-band fit.
        all_fit_x.append(x_array)
        all_fit_y.append(y_array)

        band_color = band_colors[
            wavelength
        ]

        band_marker = band_markers[
            wavelength
        ]

        is_focus_band = (
            wavelength in focus_bands
        )

        # -------------------------------------------------
        # Error bars
        # -------------------------------------------------

        ax.errorbar(
            x_values,
            y_values,
            xerr=x_errors,
            yerr=y_errors,
            fmt="none",
            ecolor=band_color,
            elinewidth=1,
            alpha=(
                0.35
                if is_focus_band
                else 0.15
            ),
            zorder=(
                3
                if is_focus_band
                else 1
            ),
        )

        # -------------------------------------------------
        # Mean points separated by sampling timing
        # -------------------------------------------------

        sampled_after_values = (
            x_values.index.get_level_values(
                "sampled_after_flight"
            )
        )

        for sampled_after_flight in [
            False,
            True,
        ]:
            sampling_mask = (
                sampled_after_values
                == sampled_after_flight
            )

            if not sampling_mask.any():
                continue

            ax.scatter(
                x_values.loc[sampling_mask],
                y_values.loc[sampling_mask],
                marker=band_marker,
                color=band_color,
                edgecolor="black",
                linewidth=0.5,
                s=MARKER_SIZES[
                    sampled_after_flight
                ],
                alpha=(
                    0.65
                    if is_focus_band
                    else 0.30
                ),
                zorder=(
                    4
                    if is_focus_band
                    else 2
                ),
            )

        # -------------------------------------------------
        # Calculate each focus-band fit
        # -------------------------------------------------

        if is_focus_band:
            fit = calculate_linear_fit(
                x_array,
                y_array,
            )

            if fit is not None:

                # Extend to x=0 to display the intercept.
                x_fit = np.linspace(
                    0,
                    max(
                        0,
                        fit["x"].max(),
                    ),
                    100,
                )

                y_fit = (
                    fit["slope"] * x_fit
                    + fit["intercept"]
                )

                # Plot individual focus-band fits only
                # when the option is enabled.
                if SHOW_INDIVIDUAL_BAND_FITS:
                    ax.plot(
                        x_fit,
                        y_fit,
                        color=band_color,
                        linestyle="-",
                        linewidth=2,
                        alpha=0.9,
                        zorder=4,
                    )

                # Store fit statistics even if the line is hidden.
                fit_results.append(
                    {
                        "site": site,
                        "wavelength": wavelength,
                        "band": band_labels[
                            wavelength
                        ],
                        "fit_type": "individual band",
                        "line_displayed": (
                            SHOW_INDIVIDUAL_BAND_FITS
                        ),
                        "n": fit["n"],
                        "slope": fit["slope"],
                        "intercept": fit["intercept"],
                        "r_squared": fit["r_squared"],
                    }
                )

    # -----------------------------------------------------
    # Combined grey fit using all wavelengths
    # -----------------------------------------------------

    if all_fit_x:
        pooled_x = np.concatenate(
            all_fit_x
        )

        pooled_y = np.concatenate(
            all_fit_y
        )

        pooled_fit = calculate_linear_fit(
            pooled_x,
            pooled_y,
        )

        if pooled_fit is not None:

            pooled_x_fit = np.linspace(
                0,
                max(
                    0,
                    pooled_fit["x"].max(),
                ),
                200,
            )

            pooled_y_fit = (
                pooled_fit["slope"]
                * pooled_x_fit
                + pooled_fit["intercept"]
            )

            # The pooled grey fit is always displayed.
            ax.plot(
                pooled_x_fit,
                pooled_y_fit,
                color="0.25",
                linestyle="--",
                linewidth=2.5,
                alpha=0.9,
                zorder=5,
            )

            fit_results.append(
                {
                    "site": site,
                    "wavelength": "all",
                    "band": "All bands combined",
                    "fit_type": "all bands",
                    "line_displayed": True,
                    "n": pooled_fit["n"],
                    "slope": pooled_fit["slope"],
                    "intercept": pooled_fit["intercept"],
                    "r_squared": pooled_fit["r_squared"],
                }
            )

    # Site label
    ax.set_title(
        config.SITE_LABELS.get(
            site,
            site,
        )
    )

    ax.grid(
        linestyle="--",
        alpha=0.2,
    )


# =========================================================
# Common limits
#
# x starts at exactly zero.
# y includes zero and any negative fitted intercept.
# =========================================================

common_x_max = max(
    0,
    *[
        ax.get_xlim()[1]
        for ax in axes
    ],
)

common_y_min = min(
    0,
    *[
        ax.get_ylim()[0]
        for ax in axes
    ],
)

common_y_max = max(
    0,
    *[
        ax.get_ylim()[1]
        for ax in axes
    ],
)


# Only add padding to the upper x-limit
x_range = common_x_max

x_padding = (
    x_range * 0.02
    if x_range > 0
    else 0.01
)


# Add a small amount of vertical padding
y_range = (
    common_y_max
    - common_y_min
)

y_padding = (
    y_range * 0.03
    if y_range > 0
    else 0.01
)


x_lower = 0

x_upper = (
    common_x_max
    + x_padding
)

y_lower = (
    common_y_min - y_padding
    if common_y_min < 0
    else 0
)

y_upper = (
    common_y_max
    + y_padding
)


for ax in axes:
    ax.set_xlim(
        x_lower,
        x_upper,
    )

    ax.set_ylim(
        y_lower,
        y_upper,
    )


# =========================================================
# Shared axis labels
# =========================================================

fig.supxlabel(
    "Ground-truth mean reflectance",
    y=0.035,
)

fig.supylabel(
    "Imagery mean reflectance",
    x=0.02,
)


# =========================================================
# Legend above the plots
# =========================================================

band_legend_handles = []

for wavelength in all_bands:

    is_focus_band = (
        wavelength in focus_bands
    )

    show_line_in_legend = (
        is_focus_band
        and SHOW_INDIVIDUAL_BAND_FITS
    )

    band_legend_handles.append(
        Line2D(
            [0],
            [0],
            marker=band_markers[
                wavelength
            ],
            linestyle=(
                "-"
                if show_line_in_legend
                else "none"
            ),
            color=band_colors[
                wavelength
            ],
            markerfacecolor=band_colors[
                wavelength
            ],
            markeredgecolor="black",
            markeredgewidth=0.5,
            linewidth=(
                2
                if show_line_in_legend
                else 0
            ),
            markersize=(
                8
                if is_focus_band
                else 6
            ),
            alpha=(
                0.9
                if is_focus_band
                else 0.5
            ),
            label=band_labels[
                wavelength
            ],
        )
    )


# Grey pooled-fit legend handle
band_legend_handles.append(
    Line2D(
        [0],
        [0],
        color="0.25",
        linestyle="--",
        linewidth=2.5,
        label="All bands combined fit",
    )
)


# Sampling/trampling-status handles
sampling_legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markerfacecolor="0.6",
        markeredgecolor="black",
        markeredgewidth=0.5,
        markersize=np.sqrt(
            MARKER_SIZES[
                sampled_after_flight
            ]
        ),
        label=SAMPLING_LABELS[
            sampled_after_flight
        ],
    )
    for sampled_after_flight in [
        False,
        True,
    ]
]


if SHOW_INDIVIDUAL_BAND_FITS:
    legend_title = (
        "Spectral band and sampling status — "
        "solid lines are band fits; dashed line is all-band fit"
    )
else:
    legend_title = (
        "Spectral band and sampling status — "
        "dashed line is all-band fit"
    )


fig.legend(
    handles=(
        band_legend_handles
        + sampling_legend_handles
    ),
    loc="upper center",
    bbox_to_anchor=(0.5, 0.99),
    ncol=6,
    title=legend_title,
)


# Manually control margins to avoid excessive space beneath
# the shared x-axis label.
fig.subplots_adjust(
    left=0.08,
    right=0.98,
    bottom=0.11,
    top=0.70,
    wspace=0.20,
)

plt.show()


# =========================================================
# Display fitted coefficients
# =========================================================

df_band_fit_results = (
    pd.DataFrame(fit_results)
    .sort_values(
        [
            "site",
            "fit_type",
            "wavelength",
        ]
    )
    .reset_index(drop=True)
)

display(df_band_fit_results)

In [ ]:

# =========================================================
# Settings
# =========================================================

red_band = "668"

all_bands = [
    str(wavelength)
    for wavelength in config.MICASENSE_10B_CENTER_WAVELENGTHS
]

non_red_bands = [
    wavelength
    for wavelength in all_bands
    if wavelength != red_band
]


# =========================================================
# Validate required columns
# =========================================================

required_columns = [
    f"{source}_{wavelength}_mean"
    for wavelength in all_bands
    for source in [
        "gt",
        "imagery",
    ]
]

missing_columns = [
    column
    for column in required_columns
    if column not in df_band_stats.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# =========================================================
# Available sites
# =========================================================

available_sites = set(
    df_band_stats.index.get_level_values("site")
)

sites = [
    site
    for site in config.SITES
    if site in available_sites
]

if not sites:
    raise ValueError(
        "None of the configured sites were found in df_band_stats."
    )


# =========================================================
# Calculate non-red fit and red-band bias for each site
# =========================================================

bias_summary = []
bias_details = []

for site in sites:

    df_site = df_band_stats.xs(
        site,
        level="site",
    )

    # -----------------------------------------------------
    # Pool all valid non-red-band observations
    # -----------------------------------------------------

    pooled_x = []
    pooled_y = []

    for wavelength in non_red_bands:

        x = pd.to_numeric(
            df_site[f"gt_{wavelength}_mean"],
            errors="coerce",
        )

        y = pd.to_numeric(
            df_site[f"imagery_{wavelength}_mean"],
            errors="coerce",
        )

        valid = (
            x.notna()
            & y.notna()
        )

        if valid.any():
            pooled_x.append(
                x.loc[valid].to_numpy(
                    dtype=float
                )
            )

            pooled_y.append(
                y.loc[valid].to_numpy(
                    dtype=float
                )
            )

    if not pooled_x:
        raise ValueError(
            f"No valid non-red-band observations were found "
            f"for site {site!r}."
        )

    pooled_x = np.concatenate(
        pooled_x
    )

    pooled_y = np.concatenate(
        pooled_y
    )

    if (
        len(pooled_x) < 2
        or np.unique(pooled_x).size < 2
    ):
        raise ValueError(
            f"Not enough distinct non-red-band observations "
            f"to fit a regression for site {site!r}."
        )

    # -----------------------------------------------------
    # Fit all bands except red
    # -----------------------------------------------------

    slope, intercept = np.polyfit(
        pooled_x,
        pooled_y,
        deg=1,
    )

    pooled_predictions = (
        slope * pooled_x
        + intercept
    )

    residual_sum_squares = np.sum(
        (
            pooled_y
            - pooled_predictions
        ) ** 2
    )

    total_sum_squares = np.sum(
        (
            pooled_y
            - pooled_y.mean()
        ) ** 2
    )

    if total_sum_squares > 0:
        fit_r_squared = (
            1
            - residual_sum_squares
            / total_sum_squares
        )
    else:
        fit_r_squared = np.nan

    # -----------------------------------------------------
    # Extract observed red-band values
    # -----------------------------------------------------

    red_gt = pd.to_numeric(
        df_site[f"gt_{red_band}_mean"],
        errors="coerce",
    )

    red_imagery = pd.to_numeric(
        df_site[f"imagery_{red_band}_mean"],
        errors="coerce",
    )

    valid_red = (
        red_gt.notna()
        & red_imagery.notna()
    )

    if not valid_red.any():
        raise ValueError(
            f"No valid red-band observations were found "
            f"for site {site!r}."
        )

    red_gt_valid = red_gt.loc[
        valid_red
    ]

    red_imagery_valid = red_imagery.loc[
        valid_red
    ]

    # -----------------------------------------------------
    # Predict red imagery reflectance from the non-red fit
    # -----------------------------------------------------

    red_predicted = (
        slope * red_gt_valid
        + intercept
    )

    # Positive means observed red imagery is higher
    # than expected from the non-red fit.
    red_bias = (
        red_imagery_valid
        - red_predicted
    )

    # Relative bias compared with predicted red reflectance
    red_relative_bias = (
        red_bias
        / red_predicted.where(
            red_predicted.ne(0)
        )
        * 100
    )

    # -----------------------------------------------------
    # Per-station details
    # -----------------------------------------------------

    detail = (
        red_gt_valid.index
        .to_frame(index=False)
    )

    detail.insert(
        0,
        "site",
        site,
    )

    detail.insert(
        1,
        "site_label",
        config.SITE_LABELS.get(
            site,
            site,
        ),
    )

    detail["red_gt_mean"] = (
        red_gt_valid.to_numpy()
    )

    detail["red_imagery_observed"] = (
        red_imagery_valid.to_numpy()
    )

    detail["red_imagery_predicted"] = (
        red_predicted.to_numpy()
    )

    detail["red_bias"] = (
        red_bias.to_numpy()
    )

    detail["red_relative_bias_percent"] = (
        red_relative_bias.to_numpy()
    )

    bias_details.append(
        detail
    )

    # -----------------------------------------------------
    # Site summary
    # -----------------------------------------------------

    bias_summary.append(
        {
            "site": site,
            "site_label": config.SITE_LABELS.get(
                site,
                site,
            ),
            "non_red_fit_n": len(pooled_x),
            "red_n": len(red_bias),
            "non_red_fit_slope": slope,
            "non_red_fit_intercept": intercept,
            "non_red_fit_r_squared": fit_r_squared,
            "red_mean_bias": red_bias.mean(),
            "red_median_bias": red_bias.median(),
            "red_bias_std": red_bias.std(ddof=1),
            "red_mean_absolute_bias": red_bias.abs().mean(),
            "red_rmse": np.sqrt(
                np.mean(
                    np.square(red_bias)
                )
            ),
            "red_mean_relative_bias_percent": (
                red_relative_bias.mean()
            ),
        }
    )


# =========================================================
# Results
# =========================================================

df_red_band_bias_summary = (
    pd.DataFrame(bias_summary)
    .set_index("site")
)

df_red_band_bias_details = (
    pd.concat(
        bias_details,
        ignore_index=True,
    )
)


print(
    "Red-band bias relative to the fit of all other bands:"
)

display(
    df_red_band_bias_summary
)

display(
    df_red_band_bias_details
)

df_red_band_bias_summary[
    [
        "site_label",
        "red_mean_bias",
        "red_mean_relative_bias_percent",
        "red_rmse",
    ]
]

# END